# exp017 — eca_nfnet_l0 SED training + Pseudo + Kaggle upload (Colab Pro A100)

**Goal:** eca_nfnet_l0 SED + Perch v2 distillation、25 epochs、Colab Pro Blackwell で 1 session 完走 → そのまま pseudo CSV + Kaggle Dataset upload まで完結。想定 ~45-60 min。

**v2 変更点 (v1 val 0.9065 → 0.91+ 狙い):**
- LR: 5.1e-4 → **3e-4** (NFNet は BN-free で sqrt scaling が強すぎた)
- ep: 20 → **25** (累計 step 3,060 → 3,825 で補填)

## なぜ eca_nfnet_l0?
- 24M params、NFNet 系 (BN-free + scaled weight standardization + ECA attention)
- HGNet (BN+ReLU+HG-block) / ConvNeXt (LN+GELU+DW) / EffNet (BN+SiLU+MBConv) / RegNet (BN+SiLU+GroupConv) と原理的 diversity
- **BC25 1位 Nikita 採用**、最も BC 実績豊富
- 4 backbone ensemble の決定打候補

## Recipe (Source mix は exp016 R1、train hyperparams は exp016 R2 ベースで Blackwell 最適化)
- focal 0.85 / labeled_sc 0.15、pseudo 無し (R1 ベース、まだ teacher pseudo なし)
- Perch v2 MSE distillation (alpha=1.0)
- **25 ep** / **batch=192** / AdamW **lr=3e-4** / cosine + warmup **3 ep** / **NUM_WORKERS=8** / **prefetch_factor=4**
- soft mixup (alpha=0.4)、SpecAugment

> ⚠️ batch=192 は Blackwell 96GB 前提。A100 40GB で実行する場合は BATCH=64-80、LR も sqrt 補正で下げること。

## NB 構成 (train + pseudo + upload 統合)
1-13. Train phase (exp016 と同じ構造、20 ep)
14-17. Pseudo phase (best_ns22 ckpt で 10,658 train_soundscapes 推論 → CSV → Drive)
18-19. Kaggle Dataset upload (weights → maekeso/birdclef2026-exp017-weights)

## Drive 構成
- input: `/content/drive/MyDrive/kaggle/birdclef2026/`
- output mirror: `/content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1/`
- pseudo:        `/content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo/`

## 出力
- `/content/output/ckpt_latest.pth`、`ckpt_ep{NN}.pth`
- `/content/output/ckpt_best_ns22.pth`、`ckpt_best_macro.pth`
- `/content/output/history.json`
- `/content/output/pseudo_labels.csv`
- すべて Drive output dir にミラー、weights は Kaggle Dataset へ upload


In [2]:
# ============================================================
# Cell 1: Setup — Drive mount, pip install, kaggle.json
# ============================================================
!pip install -q timm onnxruntime-gpu librosa soundfile scipy

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, shutil, time, subprocess
from pathlib import Path

DRIVE_INPUT_DIR  = Path("/content/drive/MyDrive/kaggle/birdclef2026")
DRIVE_OUTPUT_DIR = DRIVE_INPUT_DIR / "output" / "exp017" / "r1"
DRIVE_PSEUDO_DIR = DRIVE_INPUT_DIR / "output" / "exp017" / "r1-pseudo"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_PSEUDO_DIR.mkdir(parents=True, exist_ok=True)
assert DRIVE_INPUT_DIR.exists(), f"Drive input folder missing: {DRIVE_INPUT_DIR}"
print(f"Drive input:  {DRIVE_INPUT_DIR}")
print(f"Drive output: {DRIVE_OUTPUT_DIR}")
print(f"Drive pseudo: {DRIVE_PSEUDO_DIR}")

# kaggle.json (for Perch ONNX DL if needed + Kaggle Dataset upload)
KJ_CANDIDATES = [
    DRIVE_INPUT_DIR / "kaggle.json",
    Path("/content/drive/MyDrive/kaggle.json"),
]
KJ = next((p for p in KJ_CANDIDATES if p.exists()), None)
if KJ is not None:
    KAGGLE_CFG = Path.home() / ".kaggle"
    KAGGLE_CFG.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(KJ), str(KAGGLE_CFG / "kaggle.json"))
    os.chmod(str(KAGGLE_CFG / "kaggle.json"), 0o600)
    creds = json.loads(KJ.read_text())
    if creds.get("key", "").startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = creds["key"]
    print(f"kaggle.json: {KJ}")
else:
    print("kaggle.json not found (Perch ONNX must already be on Drive, upload will fail)")

LOCAL_DATA = Path("/content/data")
LOCAL_OUT  = Path("/content/output")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print(f"Local data: {LOCAL_DATA}")
print(f"Local out:  {LOCAL_OUT}")


Mounted at /content/drive
Drive input:  /content/drive/MyDrive/kaggle/birdclef2026
Drive output: /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1
Drive pseudo: /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo
kaggle.json: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json
Local data: /content/data
Local out:  /content/output


In [3]:
# ============================================================
# Cell 2: Data prep — Kaggle API direct DL (NO Drive copy)
# ============================================================
# Drive FUSE 個別 copy は 46k ファイルだと 1-3h かかるため、
# competition data は Kaggle API の single zip で一括取得 (5-15 min)。
import time, zipfile, subprocess
from kaggle.api.kaggle_api_extended import KaggleApi
from tqdm.auto import tqdm

api = KaggleApi(); api.authenticate()
print("kaggle authenticated")

T0_total = time.time()

TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"

need_dl = (
    not TA_DIR.exists() or sum(1 for _ in TA_DIR.rglob("*.ogg")) < 40000 or
    not TS_DIR.exists() or sum(1 for _ in TS_DIR.glob("*.ogg")) < 10000
)

if need_dl:
    print(f"\n[1/2] Downloading birdclef-2026 competition data (~25GB) via Kaggle API...")
    print(f"      tqdm progress bar will appear (size in MB, %, ETA)")
    t0 = time.time()
    api.competition_download_files(
        "birdclef-2026",
        path=str(LOCAL_DATA),
        force=False, quiet=False,
    )
    print(f"  DL done in {time.time()-t0:.0f}s ({time.time()-t0:.0f}/60={(time.time()-t0)/60:.1f} min)")

    zips = list(LOCAL_DATA.glob("birdclef-2026*.zip"))
    assert zips, "No birdclef-2026 zip found after download"
    zip_path = zips[0]
    print(f"\n[2/2] Extracting {zip_path.name} ({zip_path.stat().st_size/1e9:.1f}GB)...")
    t_extract = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        total_bytes = sum(i.file_size for i in infos)
        pbar = tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024,
                    desc="extract", smoothing=0.05, mininterval=1.0)
        for info in infos:
            zf.extract(info, LOCAL_DATA)
            pbar.update(info.file_size)
        pbar.close()
    print(f"  extracted {len(infos)} entries in {time.time()-t_extract:.0f}s "
          f"({(time.time()-t_extract)/60:.1f} min)")
    zip_path.unlink()
    print(f"  removed zip")
else:
    print("Competition data already present locally")

# Verify
n_ta = sum(1 for _ in TA_DIR.rglob("*.ogg")) if TA_DIR.exists() else 0
n_ts = sum(1 for _ in TS_DIR.glob("*.ogg")) if TS_DIR.exists() else 0
print(f"\n  train_audio:       {n_ta} .ogg files")
print(f"  train_soundscapes: {n_ts} .ogg files")

# Place CSVs under /content/data/competition (rest of NB expects this layout)
comp = LOCAL_DATA / "competition"
comp.mkdir(parents=True, exist_ok=True)
for fn in ["train.csv", "taxonomy.csv", "sample_submission.csv", "train_soundscapes_labels.csv"]:
    src_in_root = LOCAL_DATA / fn
    dst = comp / fn
    if src_in_root.exists() and not dst.exists():
        shutil.copy2(str(src_in_root), str(dst))
        print(f"  competition/{fn} OK")
    elif not src_in_root.exists():
        drive_src = DRIVE_INPUT_DIR / fn
        if drive_src.exists() and not dst.exists():
            shutil.copy2(str(drive_src), str(dst))
            print(f"  competition/{fn} OK (from Drive fallback)")

# Perch ONNX (Drive 優先、無ければ Kaggle DL)
po_dir = LOCAL_DATA / "perch-onnx"
po_dir.mkdir(parents=True, exist_ok=True)
PERCH_PATH = po_dir / "perch_v2_no_dft.onnx"

if not PERCH_PATH.exists():
    drive_perch_candidates = list(DRIVE_INPUT_DIR.rglob("perch_v2*.onnx"))
    if drive_perch_candidates:
        src = drive_perch_candidates[0]
        sz = src.stat().st_size
        print(f"\nCopying Perch ONNX from Drive: {src.name} ({sz/1e9:.2f}GB)")
        t0 = time.time()
        with open(src, "rb") as fin, open(PERCH_PATH, "wb") as fout:
            pbar = tqdm(total=sz, unit="B", unit_scale=True, unit_divisor=1024,
                        desc="Perch copy", mininterval=1.0)
            while True:
                buf = fin.read(8 * 1024 * 1024)
                if not buf: break
                fout.write(buf); pbar.update(len(buf))
            pbar.close()
        print(f"  done in {time.time()-t0:.0f}s")
    else:
        print("\nPerch ONNX not on Drive, downloading from Kaggle...")
        t0 = time.time()
        api.dataset_download_files(
            "tuckerarrants/perch-v2-no-dft-onnx",
            path=str(po_dir), unzip=True, quiet=False)
        print(f"  done in {time.time()-t0:.0f}s")
        if not PERCH_PATH.exists():
            hits = list(po_dir.rglob("perch_v2*.onnx"))
            if hits and hits[0] != PERCH_PATH:
                shutil.move(str(hits[0]), str(PERCH_PATH))
assert PERCH_PATH.exists(), f"Perch ONNX missing: {PERCH_PATH}"
print(f"Perch ONNX: {PERCH_PATH} ({PERCH_PATH.stat().st_size/1e9:.2f} GB)")

# Disk summary
print(f"\n=== Total DL time: {(time.time()-T0_total)/60:.1f} min ===")
r = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(r.stdout)


kaggle authenticated

[1/2] Downloading birdclef-2026 competition data (~25GB) via Kaggle API...
      tqdm progress bar will appear (size in MB, %, ETA)


100%|██████████| 15.0G/15.0G [06:18<00:00, 42.5MB/s]



  DL done in 378s (378/60=6.3 min)

[2/2] Extracting birdclef-2026.zip (16.1GB)...


extract:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

  extracted 46213 entries in 63s (1.1 min)
  removed zip

  train_audio:       35549 .ogg files
  train_soundscapes: 10658 .ogg files
  competition/train.csv OK
  competition/taxonomy.csv OK
  competition/sample_submission.csv OK
  competition/train_soundscapes_labels.csv OK

Perch ONNX not on Drive, downloading from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/perch-v2-no-dft-onnx


100%|██████████| 431M/431M [00:11<00:00, 39.4MB/s]



  done in 14s
Perch ONNX: /content/data/perch-onnx/perch_v2_no_dft.onnx (0.41 GB)

=== Total DL time: 8.4 min ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   60G  177G  26% /



In [4]:
# ============================================================
# Cell 3: Imports + Config + Paths
# ============================================================
import os, time, json, gc, random, math
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
import warnings
warnings.filterwarnings("ignore")

# ===== Paths =====
BASE = LOCAL_DATA / "competition"
TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"
TAXO_PATH = BASE / "taxonomy.csv"
TRAIN_CSV = BASE / "train.csv"
SAMPLE_SUB_PATH = BASE / "sample_submission.csv"
LABELS_PATH = BASE / "train_soundscapes_labels.csv"
PERCH_PATH = LOCAL_DATA / "perch-onnx" / "perch_v2_no_dft.onnx"
OUT_DIR = LOCAL_OUT
print(f"BASE: {BASE}")
print(f"TA_DIR: {TA_DIR.exists()}, TS_DIR: {TS_DIR.exists()}")
print(f"PERCH: {PERCH_PATH.exists()}")

# ===== Reproducibility =====
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ===== Config =====
NUM_CLASSES = 234
SR = 32000
TRAIN_DURATION = 5
VAL_DURATION   = 5
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION
N_FFT          = 2048
HOP_LENGTH     = 512
N_MELS         = 256
FMIN           = 20
FMAX           = 16000

BACKBONE = "eca_nfnet_l0"

USE_PERCH_DISTILL = True
PERCH_EMBED_DIM = 1536
ALPHA_DISTILL = 1.0

N_FOLDS = 5
FOLDS = [0]
FORCE_FRESH = True        # ★ v2 で LR/ep 変更したので Drive 上の v1 ckpt は無視して新規学習
# ★ R1 v2: NFNet 用に LR を保守 (3e-4)、step 数を稼ぐため ep=25
# 元 r1 v1 (LR=5.1e-4, ep=20) は val 0.9065 で plateau — NFNet (BN-free) に sqrt scaling は強すぎ
# 注: A100 40GB で実行する場合は BATCH=64-80 まで下げる必要あり (LR は同じ)
N_TOTAL_EPOCHS = 25       # ★ v1 比 +5、累計 step 3,060 → 3,825 で step 補填
BATCH = 192               # ★ Blackwell 96GB の VRAM 活用 (v1 と同じ)
LR = 3e-4                 # ★ v1 の 5.1e-4 → 3e-4、NFNet は AGC 無しなら保守 LR が定石
MIN_LR = 1e-6
WD = 1e-4
WARMUP_EPOCHS = 3

AUG_PROB = 0.5
AUG_GAIN_DB_RANGE = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

USE_MIXUP = True
MIXUP_PROB = 0.5
MIXUP_ALPHA = 0.4
MIXUP_HARD = False

FREQ_MASK_PARAM = 25
TIME_MASK_PARAM = 30
NUM_FREQ_MASKS = 2
NUM_TIME_MASKS = 2

MIN_SAMPLE = 20

SHARES = {"focal": 0.85, "sc": 0.15}
SOURCE_WEIGHTS = {"focal": 1.0, "focal_missing": 0.0, "sc": 1.0}

NUM_WORKERS = 8           # ★ exp016 R2 と同じ (4→8 で +20-40%、batch=128 のデータ供給用)
PERSISTENT_WORKERS = True

# ===== Timer (Colab Pro: A100 typically 24h limit, leave large buffer) =====
TRAIN_START = time.time()
MAX_RUNTIME_SEC = 22.0 * 3600   # 22h safety margin
print(f"Backbone: {BACKBONE}")
print(f"Batch: {BATCH} | Total epochs: {N_TOTAL_EPOCHS} | Folds: {FOLDS}")
print(f"LR: {LR} | WD: {WD} | warmup: {WARMUP_EPOCHS}ep")
print(f"Max runtime: {MAX_RUNTIME_SEC/3600:.1f}h")


BASE: /content/data/competition
TA_DIR: True, TS_DIR: True
PERCH: True
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
Backbone: eca_nfnet_l0
Batch: 192 | Total epochs: 25 | Folds: [0]
LR: 0.0003 | WD: 0.0001 | warmup: 3ep
Max runtime: 22.0h


In [5]:
# ============================================================
# Cell 4: Load data — train.csv, taxonomy, labeled SS, fold assignment
# ============================================================
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
assert len(PRIMARY_LABELS) == NUM_CLASSES

taxonomy = pd.read_csv(TAXO_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df[train_df["primary_label"].astype(str).isin(LABEL2IDX)].reset_index(drop=True)
train_df["filename"] = train_df["filename"].astype(str)
print(f"Focal train.csv: {len(train_df)} rows")

def _check_exists(fn):
    return (TA_DIR / fn).exists()
print("Checking focal file existence...")
_t0 = time.time()
train_df["exists"] = train_df["filename"].map(_check_exists)
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print(f"  {len(train_df)} focal files exist ({time.time()-_t0:.1f}s)")
train_df["original_idx"] = np.arange(len(train_df))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    train_df.loc[val_idx, "fold"] = fold
print(f"Focal fold distribution: {train_df['fold'].value_counts().sort_index().to_dict()}")

focal_secondary_labels = {}
for idx, row in train_df.iterrows():
    sec = row.get("secondary_labels", "")
    if pd.isna(sec) or sec in ("", "[]"):
        continue
    try:
        sec_list = eval(sec) if isinstance(sec, str) else []
    except Exception:
        continue
    valid = [s for s in sec_list if s in LABEL2IDX]
    if valid:
        focal_secondary_labels[int(row["original_idx"])] = valid
print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

counts = train_df["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index.tolist()
extra_rows = []
for sp in rare_species:
    sp_rows = train_df[train_df["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)
n_before = len(train_df)
if extra_rows:
    train_df = pd.concat([train_df] + extra_rows, ignore_index=True)
print(f"Upsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): {n_before} -> {len(train_df)}")

if LABELS_PATH.exists():
    sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
    if sc_labels_raw["start"].dtype == object:
        sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)
    else:
        sc_labels_raw["start_sec"] = sc_labels_raw["start"].astype(int)

    sc_meta = (sc_labels_raw[["filename", "start_sec"]]
               .drop_duplicates()
               .reset_index(drop=True))
    if "site" in sc_labels_raw.columns:
        site_map = sc_labels_raw.groupby("filename")["site"].first().to_dict()
        sc_meta["site"] = sc_meta["filename"].map(site_map).fillna("UNK")
    else:
        sc_meta["site"] = "UNK"

    Y_SC = np.zeros((len(sc_meta), NUM_CLASSES), dtype=np.float32)
    for i, row in sc_meta.iterrows():
        matches = sc_labels_raw[(sc_labels_raw["filename"] == row["filename"]) &
                                 (sc_labels_raw["start_sec"] == row["start_sec"])]
        for _, m in matches.iterrows():
            for lbl in str(m["primary_label"]).split(";"):
                lbl = lbl.strip()
                if lbl in LABEL2IDX:
                    Y_SC[i, LABEL2IDX[lbl]] = 1.0
    print(f"Soundscape labels: {len(sc_meta)} windows, {int(Y_SC.sum())} positives, "
          f"{int((Y_SC.sum(axis=0) > 0).sum())} species")

    sc_files = sc_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
    gkf = GroupKFold(n_splits=N_FOLDS)
    sc_files["fold"] = -1
    for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
        sc_files.loc[sc_files.index[val_idx], "fold"] = fold
    file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
    sc_meta["fold"] = sc_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
    print(f"SS fold distribution: {sc_meta['fold'].value_counts().sort_index().to_dict()}")

    non_s22_mask_sc = (sc_meta["site"].values != "S22")
    print(f"Non-S22: {non_s22_mask_sc.sum()}/{len(sc_meta)}")
else:
    print("LABELS_PATH missing — labeled SS source disabled")
    sc_meta = pd.DataFrame(columns=["filename", "start_sec", "site", "fold"])
    Y_SC = np.zeros((0, NUM_CLASSES), dtype=np.float32)
    non_s22_mask_sc = np.zeros(0, dtype=bool)

print("OK data loaded")


Focal train.csv: 35549 rows
Checking focal file existence...
  35549 focal files exist (0.2s)
Focal fold distribution: {0: 7110, 1: 7110, 2: 7110, 3: 7110, 4: 7109}
Focal secondary labels: 4372 files
Upsampled 36 rare species (min=20): 35549 -> 36135
Soundscape labels: 739 windows, 3122 positives, 75 species
SS fold distribution: {0: 155, 1: 137, 2: 146, 3: 149, 4: 152}
Non-S22: 739/739
OK data loaded


In [6]:
# ============================================================
# Cell 5: Model — Mel + SpecAugment + Perch teacher + DistillHead + BirdSEDModel
# ============================================================
import onnxruntime as ort

class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS): mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS): mel = self.time_mask(mel)
        return mel

class PerchTeacher:
    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i; break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"PerchTeacher: embed_idx={self._embed_idx}, providers={self.session.get_providers()}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        wav_np = waveforms_5s.cpu().numpy().astype(np.float32)
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])
        return self.proj(gap)

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"Backbone out: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, "distill_head"):
            distill_emb = self.distill_head(h)
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill:
            return clip_logits, fw, distill_emb
        elif return_framewise:
            return clip_logits, fw
        elif return_distill:
            return clip_logits, distill_emb
        return clip_logits

def make_model():
    m = BirdSEDModel().to(device)
    m = m.to(memory_format=torch.channels_last)
    return m

print("OK model defs ready")


OK model defs ready


In [7]:
# ============================================================
# Cell 6: Datasets — FocalDS, LabeledSCDS, augmentation, samplers
# ============================================================
import soundfile as sf
import librosa

def _load_ogg(path, n_samples_target, start_sec=None):
    try:
        if start_sec is not None:
            wav, sr = sf.read(str(path),
                              start=int(start_sec * SR),
                              frames=n_samples_target,
                              dtype="float32", always_2d=False)
        else:
            wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        return wav.astype(np.float32)
    except Exception:
        return None

def extract_chunk_np(waveform, start_sample, n_samples):
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w


class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, aug=False):
        self.df = df.reset_index(drop=True)
        self.l2i = l2i
        self.aug = aug
        self.secondary_lookup = secondary_lookup
        self.filenames = self.df["filename"].values
        self.primary = self.df["primary_label"].astype(str).values
        self.original_idx = self.df["original_idx"].values if "original_idx" in self.df.columns else None

    def __len__(self): return len(self.df)

    def _load_chunk(self, i):
        fn = self.filenames[i]
        path = TA_DIR / fn
        wav = _load_ogg(path, n_samples_target=None)
        if wav is None:
            return None, None
        if self.aug and len(wav) > TRAIN_SAMPLES:
            start = np.random.randint(0, len(wav) - TRAIN_SAMPLES + 1)
        else:
            start = 0
        chunk = extract_chunk_np(wav, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if self.primary[i] in self.l2i:
            lb[self.l2i[self.primary[i]]] = 1.0
        if self.secondary_lookup is not None and self.original_idx is not None:
            for s in self.secondary_lookup.get(int(self.original_idx[i]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return chunk, lb

    def __getitem__(self, i):
        ch1, lb1 = self._load_chunk(i)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing")

        if USE_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2, lb2 = None, None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(j)
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0),
                        torch.from_numpy(lb.astype(np.float32)),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")


class LabeledSCDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y = Y
        self.df = sc_df.reset_index(drop=True)
        self.aug = aug
        self.filenames = self.df["filename"].astype(str).values
        self.start_secs = self.df["start_sec"].astype(int).values

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        fn = self.filenames[i]
        if not fn.endswith(".ogg"): fn = fn + ".ogg"
        path = TS_DIR / fn
        wav = _load_ogg(path, n_samples_target=TRAIN_SAMPLES,
                         start_sec=float(self.start_secs[i]))
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        if len(wav) < TRAIN_SAMPLES:
            wav = np.pad(wav, (0, TRAIN_SAMPLES - len(wav)))
        else:
            wav = wav[:TRAIN_SAMPLES]
        if self.aug:
            wav = apply_aug(wav)
        return (torch.from_numpy(wav.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc")


class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch])

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK datasets ready")


OK datasets ready


In [8]:
# ============================================================
# Cell 7: Eval — AUC + validation predictor
# ============================================================
def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)


def full_eval(y_true, y_pred, ns22_mask, taxon_masks):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["n_all"] = n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask)
    r["non_s22_macro"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["n_ns22"] = n
    per_taxon = {}
    for t, cm in taxon_masks.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask, class_mask=cm)
        per_taxon[t] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["per_taxon"] = per_taxon
    return r


def _load_val_waveforms(val_sc_df):
    wavs = []
    for _, row in val_sc_df.iterrows():
        fn = str(row["filename"])
        if not fn.endswith(".ogg"): fn = fn + ".ogg"
        wav = _load_ogg(TS_DIR / fn, n_samples_target=VAL_SAMPLES,
                         start_sec=float(row["start_sec"]))
        if wav is None or len(wav) == 0:
            wav = np.zeros(VAL_SAMPLES, dtype=np.float32)
        if len(wav) < VAL_SAMPLES:
            wav = np.pad(wav, (0, VAL_SAMPLES - len(wav)))
        else:
            wav = wav[:VAL_SAMPLES]
        wavs.append(torch.from_numpy(wav.astype(np.float32)).unsqueeze(0))
    return wavs


def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            mel = mel.to(memory_format=torch.channels_last)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).float().cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip),
            "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

print("OK eval helpers ready")


OK eval helpers ready


In [9]:
# ============================================================
# Cell 8: build_active_datasets per fold
# ============================================================
def build_active_datasets(fold_k):
    items = []
    fds = FocalDS(train_df[train_df["fold"] != fold_k],
                  LABEL2IDX, secondary_lookup=focal_secondary_labels, aug=True)
    items.append(("focal", fds, len(fds)))
    if len(sc_meta) > 0:
        vm = sc_meta["fold"].values == fold_k
        sc_train_df = sc_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = LabeledSCDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))
    return items

print("OK build_active_datasets ready")


OK build_active_datasets ready


In [10]:
# ============================================================
# Cell 9: init — fresh start, optionally resume from Drive ckpt_latest.pth
# ============================================================
FOLD_K = FOLDS[0]
print(f"This run trains fold {FOLD_K}")

active = build_active_datasets(FOLD_K)
NAMES, DATASETS, SIZES = zip(*active)
NAMES, DATASETS, SIZES = list(NAMES), list(DATASETS), list(SIZES)
print(f"Streams: {dict(zip(NAMES, SIZES))}")

mds = ConcatDataset(DATASETS)
N_STEPS_PER_EP = max(100, int(sum(SIZES) / BATCH))
print(f"steps/epoch: {N_STEPS_PER_EP}")

model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = GradScaler()

warmup_steps = N_STEPS_PER_EP * WARMUP_EPOCHS
total_steps  = N_STEPS_PER_EP * N_TOTAL_EPOCHS
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1/25, end_factor=1.0, total_iters=warmup_steps)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps - warmup_steps, eta_min=MIN_LR)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

start_epoch = 0
best_ns22 = -1.0
best_macro = -1.0
history = []
resumed = False

# Optional resume from Drive
DRIVE_LATEST = DRIVE_OUTPUT_DIR / "ckpt_latest.pth"
if FORCE_FRESH and DRIVE_LATEST.exists():
    print(f"\nFORCE_FRESH=True: ignoring existing ckpt {DRIVE_LATEST}, starting from scratch")
    print(f"(LR/ep changed for v2 — resume would inherit old optimizer/scheduler state)")
elif DRIVE_LATEST.exists():
    print(f"\nFound Drive ckpt: {DRIVE_LATEST}")
    try:
        state = torch.load(str(DRIVE_LATEST), map_location=device, weights_only=False)
    except TypeError:
        state = torch.load(str(DRIVE_LATEST), map_location=device)
    model.load_state_dict(state["model_state"])
    optimizer.load_state_dict(state["optimizer_state"])
    scheduler.load_state_dict(state["scheduler_state"])
    scaler.load_state_dict(state["scaler_state"])
    start_epoch = int(state["epoch"])
    best_ns22 = float(state.get("best_ns22", -1.0))
    best_macro = float(state.get("best_macro", -1.0))
    history = state.get("history", [])
    resumed = True
    print(f"=== RESUMED from epoch {start_epoch}/{N_TOTAL_EPOCHS} ===")
    print(f"    best_ns22={best_ns22:.4f}, best_macro={best_macro:.4f}, history_len={len(history)}")
    src = DRIVE_OUTPUT_DIR / "history.json"
    if src.exists():
        shutil.copy(str(src), str(OUT_DIR / "history.json"))
else:
    print("\nNo Drive ckpt; starting fresh")

if start_epoch >= N_TOTAL_EPOCHS:
    print(f"\nAlready trained to epoch {start_epoch}/{N_TOTAL_EPOCHS}. Nothing to do.")


This run trains fold 0
Streams: {'focal': 28906, 'sc': 584}
steps/epoch: 153


model.safetensors:   0%|          | 0.00/96.6M [00:00<?, ?B/s]

Backbone out: (1, 2304, 8, 10)  (C=2304)

FORCE_FRESH=True: ignoring existing ckpt /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1/ckpt_latest.pth, starting from scratch
(LR/ep changed for v2 — resume would inherit old optimizer/scheduler state)


In [11]:
# ============================================================
# Cell 10: Val prep — load val waveforms once
# ============================================================
if len(sc_meta) > 0:
    vm = sc_meta["fold"].values == FOLD_K
    val_sc_df = sc_meta[vm].reset_index(drop=True)
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    print(f"Val: {len(val_sc_df)} SS windows for fold {FOLD_K}")
    print(f"  loading val waveforms...")
    _t0 = time.time()
    val_wavs = _load_val_waveforms(val_sc_df)
    print(f"  loaded in {time.time()-_t0:.1f}s")
else:
    val_sc_df = pd.DataFrame()
    Y_val = np.zeros((0, NUM_CLASSES), dtype=np.float32)
    ns22_val = np.zeros(0, dtype=bool)
    val_wavs = []
    print("No labeled SS for validation; metrics will be NaN")


Val: 155 SS windows for fold 0
  loading val waveforms...
  loaded in 0.5s


In [12]:
# ============================================================
# Cell 11: Training loop — Drive mirror per epoch
# ============================================================
mel_transform = MelSpecTransform().to(device)
spec_augment = SpecAugment().to(device)
perch_teacher = PerchTeacher(PERCH_PATH, "cuda" if torch.cuda.is_available() else "cpu") \
                if USE_PERCH_DISTILL else None

def mirror_to_drive(local_path: Path, dst_name: str = None):
    '''Copy a single file from /content/output to DRIVE_OUTPUT_DIR. Best-effort.'''
    try:
        name = dst_name or local_path.name
        shutil.copy2(str(local_path), str(DRIVE_OUTPUT_DIR / name))
    except Exception as e:
        print(f"  [WARN] mirror_to_drive failed for {local_path.name}: {e}")

epoch_times = []
trained_this_session = 0
ep = start_epoch

for epoch in range(start_epoch, N_TOTAL_EPOCHS):
    elapsed = time.time() - TRAIN_START
    if epoch_times:
        est_next = max(epoch_times[-3:])
        if elapsed + est_next * 1.3 > MAX_RUNTIME_SEC:
            print(f"\n[stop] Time budget exhausted before epoch {epoch+1}/{N_TOTAL_EPOCHS}")
            break

    ep_start = time.time()
    model.train()

    smp = MixSamp(SIZES, NAMES, SHARES, BATCH, N_STEPS_PER_EP, seed=42 + epoch)
    train_loader = DataLoader(
        mds, batch_sampler=smp, collate_fn=collate_m,
        num_workers=NUM_WORKERS,
        persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
        pin_memory=True, prefetch_factor=4 if NUM_WORKERS > 0 else None,
    )

    el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
    for batch_idx, (wav, lb, wt, mk, sr) in enumerate(train_loader):
        wav, lb, wt, mk = wav.to(device, non_blocking=True), lb.to(device, non_blocking=True), \
                          wt.to(device, non_blocking=True), mk.to(device, non_blocking=True)
        sw = mk_sw(sr).to(device, non_blocking=True)

        with torch.no_grad():
            mel = mel_transform(wav)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            mel = spec_augment(mel)
            mel = mel.to(memory_format=torch.channels_last)

        with autocast():
            if USE_PERCH_DISTILL:
                clip_logits, framewise, distill_emb = model(mel, return_framewise=True,
                                                              return_distill=True)
            else:
                clip_logits, framewise = model(mel, return_framewise=True)

            frame_max_logits = framewise.max(dim=1).values
            bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
            bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
            bce = 0.5 * bce_clip + 0.5 * bce_frame
            ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
            cls_loss = (ps * sw).mean()

            if USE_PERCH_DISTILL and perch_teacher is not None:
                with torch.no_grad():
                    wav_5s = wav.squeeze(1)
                    N = wav_5s.shape[1]
                    if N > 160000:
                        start_off = (N - 160000) // 2
                        wav_5s = wav_5s[:, start_off:start_off + 160000]
                    elif N < 160000:
                        wav_5s = F.pad(wav_5s, (0, 160000 - N))
                    perch_emb = perch_teacher.embed(wav_5s).to(device)
                distill_loss = F.mse_loss(distill_emb, perch_emb)
                loss = cls_loss + ALPHA_DISTILL * distill_loss
            else:
                distill_loss = torch.tensor(0.0, device=device)
                loss = cls_loss

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        el += float(loss.item())
        el_cls += float(cls_loss.item())
        el_dist += float(distill_loss.item())
        nb_count += 1

        if batch_idx % 20 == 0:
            cur_lr = optimizer.param_groups[0]["lr"]
            print(f"    ep{epoch+1:02d} batch {batch_idx:4d}/{N_STEPS_PER_EP}  "
                  f"loss={loss.item():.4f}  cls={cls_loss.item():.4f}  "
                  f"dist={distill_loss.item():.4f}  lr={cur_lr:.2e}", flush=True)

    train_loss_avg = el / max(nb_count, 1)
    cls_loss_avg = el_cls / max(nb_count, 1)
    dist_loss_avg = el_dist / max(nb_count, 1)

    if len(val_wavs) > 0:
        val_preds_dict = _predict_from_waveforms(model, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        val_metrics = {
            "ns22": r["non_s22_macro"],
            "macro": r["macro_auc_all"],
            "per_taxon": r["per_taxon"],
        }
    else:
        val_metrics = {"ns22": float("nan"), "macro": float("nan"), "per_taxon": {}}

    ep_elapsed = time.time() - ep_start
    epoch_times.append(ep_elapsed)
    trained_this_session += 1
    ep = epoch + 1

    state_to_save = {
        "epoch": ep,
        "fold": FOLD_K,
        "n_total_epochs": N_TOTAL_EPOCHS,
        "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_ns22": best_ns22,
        "best_macro": best_macro,
        "history": history,
    }
    torch.save(state_to_save, OUT_DIR / "ckpt_latest.pth")
    torch.save(state_to_save, OUT_DIR / f"ckpt_ep{ep:02d}.pth")

    is_best_ns22 = (not math.isnan(val_metrics["ns22"])) and val_metrics["ns22"] > best_ns22
    is_best_macro = (not math.isnan(val_metrics["macro"])) and val_metrics["macro"] > best_macro
    if is_best_ns22:
        best_ns22 = val_metrics["ns22"]
        state_to_save["best_ns22"] = best_ns22
        torch.save(state_to_save, OUT_DIR / "ckpt_best_ns22.pth")
    if is_best_macro:
        best_macro = val_metrics["macro"]
        state_to_save["best_macro"] = best_macro
        torch.save(state_to_save, OUT_DIR / "ckpt_best_macro.pth")

    cur_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": ep,
        "train_loss": round(train_loss_avg, 5),
        "cls_loss": round(cls_loss_avg, 5),
        "dist_loss": round(dist_loss_avg, 5),
        "val_ns22": val_metrics["ns22"],
        "val_macro": val_metrics["macro"],
        "val_per_taxon": val_metrics["per_taxon"],
        "lr": cur_lr,
        "elapsed_sec": round(ep_elapsed, 1),
    })
    with open(OUT_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2, default=str)

    mirror_to_drive(OUT_DIR / "ckpt_latest.pth")
    mirror_to_drive(OUT_DIR / "history.json")
    if is_best_ns22:
        mirror_to_drive(OUT_DIR / "ckpt_best_ns22.pth")
    if is_best_macro:
        mirror_to_drive(OUT_DIR / "ckpt_best_macro.pth")

    total_elapsed = time.time() - TRAIN_START
    print(f"\n=== Ep {ep}/{N_TOTAL_EPOCHS}: "
          f"train_loss={train_loss_avg:.4f} cls={cls_loss_avg:.4f} dist={dist_loss_avg:.4f} "
          f"val_ns22={val_metrics['ns22']:.4f} val_macro={val_metrics['macro']:.4f} "
          f"({ep_elapsed:.0f}s, total {total_elapsed/60:.1f}min) ===\n")

    gc.collect()
    torch.cuda.empty_cache()

trained_up_to = ep
print(f"\n=== Session done: trained epochs {start_epoch+1}-{trained_up_to} this session ===")
print(f"    Cumulative: {trained_up_to}/{N_TOTAL_EPOCHS} epochs done")


PerchTeacher: embed_idx=0, providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    ep01 batch    0/153  loss=1.0265  cls=0.9966  dist=0.0299  lr=1.26e-05
    ep01 batch   20/153  loss=0.7895  cls=0.7753  dist=0.0143  lr=2.52e-05
    ep01 batch   40/153  loss=0.7281  cls=0.7160  dist=0.0121  lr=3.77e-05
    ep01 batch   60/153  loss=0.6824  cls=0.6709  dist=0.0115  lr=5.03e-05
    ep01 batch   80/153  loss=0.6501  cls=0.6387  dist=0.0113  lr=6.28e-05
    ep01 batch  100/153  loss=0.5863  cls=0.5750  dist=0.0114  lr=7.54e-05
    ep01 batch  120/153  loss=0.4991  cls=0.4879  dist=0.0112  lr=8.79e-05
    ep01 batch  140/153  loss=0.4081  cls=0.3964  dist=0.0117  lr=1.00e-04

=== Ep 1/25: train_loss=0.6403 cls=0.6277 dist=0.0127 val_ns22=0.5180 val_macro=0.5180 (144s, total 2.8min) ===

    ep02 batch    0/153  loss=0.3962  cls=0.3850  dist=0.0112  lr=1.09e-04
    ep02 batch   20/153  loss=0.3855  cls=0.3743  dist=0.0112  lr=1.21e-04
    ep02 batch   40/153  loss=0.3636  cls=0.3523 

In [13]:
# ============================================================
# Cell 12: Final Drive mirror — all per-epoch ckpts
# ============================================================
print("Mirroring all artifacts to Drive...")
n_copied = 0
for f in sorted(OUT_DIR.glob("*")):
    if not f.is_file(): continue
    if f.suffix not in {".pth", ".json", ".txt"}: continue
    try:
        shutil.copy2(str(f), str(DRIVE_OUTPUT_DIR / f.name))
        n_copied += 1
        print(f"  {f.name}  {f.stat().st_size/1e6:.1f} MB")
    except Exception as e:
        print(f"  [WARN] {f.name}: {e}")
print(f"\nMirrored {n_copied} files to {DRIVE_OUTPUT_DIR}")


Mirroring all artifacts to Drive...
  ckpt_best_macro.pth  321.8 MB
  ckpt_best_ns22.pth  321.8 MB
  ckpt_ep01.pth  321.8 MB
  ckpt_ep02.pth  321.8 MB
  ckpt_ep03.pth  321.8 MB
  ckpt_ep04.pth  321.8 MB
  ckpt_ep05.pth  321.8 MB
  ckpt_ep06.pth  321.8 MB
  ckpt_ep07.pth  321.8 MB
  ckpt_ep08.pth  321.8 MB
  ckpt_ep09.pth  321.8 MB
  ckpt_ep10.pth  321.8 MB
  ckpt_ep11.pth  321.8 MB
  ckpt_ep12.pth  321.8 MB
  ckpt_ep13.pth  321.8 MB
  ckpt_ep14.pth  321.8 MB
  ckpt_ep15.pth  321.8 MB
  ckpt_ep16.pth  321.8 MB
  ckpt_ep17.pth  321.8 MB
  ckpt_ep18.pth  321.8 MB
  ckpt_ep19.pth  321.8 MB
  ckpt_ep20.pth  321.8 MB
  ckpt_ep21.pth  321.8 MB
  ckpt_ep22.pth  321.8 MB
  ckpt_ep23.pth  321.8 MB
  ckpt_ep24.pth  321.8 MB
  ckpt_ep25.pth  321.8 MB
  ckpt_latest.pth  321.8 MB
  history.json  0.0 MB

Mirrored 29 files to /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1


In [14]:
# ============================================================
# Cell 13: Train session summary
# ============================================================
total_time = time.time() - TRAIN_START
print(f"\n{'='*60}")
print(f"Train session summary")
print(f"{'='*60}")
print(f"  Resume status:        {'RESUMED' if resumed else 'FRESH START'}")
print(f"  Resumed from epoch:   {start_epoch}")
print(f"  Trained this session: {trained_this_session} epoch(s)")
print(f"  Cumulative:           {trained_up_to}/{N_TOTAL_EPOCHS} epochs")
print(f"  Best ns22 AUC:        {best_ns22:.4f}")
print(f"  Best macro AUC:       {best_macro:.4f}")
print(f"  Total session time:   {total_time/60:.1f} min")
print(f"  Drive output:         {DRIVE_OUTPUT_DIR}")
if trained_up_to >= N_TOTAL_EPOCHS:
    print(f"\n  >>> Training COMPLETE — proceeding to Pseudo phase <<<")
else:
    print(f"\n  Training NOT complete ({trained_up_to}/{N_TOTAL_EPOCHS}). "
          f"Re-run this NB to continue, then run pseudo phase.")



Train session summary
  Resume status:        FRESH START
  Resumed from epoch:   0
  Trained this session: 25 epoch(s)
  Cumulative:           25/25 epochs
  Best ns22 AUC:        0.9166
  Best macro AUC:       0.9166
  Total session time:   43.4 min
  Drive output:         /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1

  >>> Training COMPLETE — proceeding to Pseudo phase <<<


---
# Phase 2: Pseudo label generation

R1 student (eca_nfnet_l0 SED) で 10,658 train_soundscapes に推論 → Power Transform → CSV → Drive 保存。


In [15]:
# ============================================================
# Cell 14: VRAM cleanup before pseudo phase
# ============================================================
# Train phase の objects を解放
try:
    del perch_teacher
except NameError:
    pass
try:
    del optimizer, scheduler, scaler, train_loader, smp, mds
except NameError:
    pass
try:
    del val_wavs
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Load best_ns22 ckpt into model (keep model around, just overwrite weights)
BEST_CKPT_LOCAL = OUT_DIR / "ckpt_best_ns22.pth"
BEST_CKPT_DRIVE = DRIVE_OUTPUT_DIR / "ckpt_best_ns22.pth"
PSEUDO_CKPT = BEST_CKPT_LOCAL if BEST_CKPT_LOCAL.exists() else BEST_CKPT_DRIVE
assert PSEUDO_CKPT.exists(), f"No best ckpt found at {BEST_CKPT_LOCAL} or {BEST_CKPT_DRIVE}"
print(f"Loading best ckpt for pseudo: {PSEUDO_CKPT}")

try:
    state_p = torch.load(str(PSEUDO_CKPT), map_location=device, weights_only=False)
except TypeError:
    state_p = torch.load(str(PSEUDO_CKPT), map_location=device)
model.load_state_dict(state_p["model_state"])
model.eval()
print(f"  epoch={state_p.get('epoch')}, best_ns22={state_p.get('best_ns22'):.4f}, "
      f"best_macro={state_p.get('best_macro'):.4f}")

PSEUDO_START = time.time()


Loading best ckpt for pseudo: /content/output/ckpt_best_ns22.pth
  epoch=24, best_ns22=0.9166, best_macro=0.9165


In [16]:
# ============================================================
# Cell 15: Inference on all train_soundscapes
# ============================================================
import glob as _glob
from scipy.ndimage import gaussian_filter1d

N_WINDOWS = 12
CHUNK_N   = TRAIN_SAMPLES
GAUSS_SIGMA  = 0.65
POWER_GAMMA  = 1.2

def load_audio_32k_mono(path, target_samples=60 * SR):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    if len(wav) < target_samples:
        wav = np.pad(wav, (0, target_samples - len(wav)))
    elif len(wav) > target_samples:
        wav = wav[:target_samples]
    return wav.astype(np.float32)


def file_to_chunks(path):
    wav = load_audio_32k_mono(path, target_samples=N_WINDOWS * CHUNK_N)
    chunks = wav.reshape(N_WINDOWS, CHUNK_N)
    return chunks.astype(np.float32)


sc_files = sorted(_glob.glob(str(TS_DIR / "*.ogg")))
print(f"train_soundscapes: {len(sc_files)} files")
assert len(sc_files) > 0

all_filenames = []
all_start_secs = []
all_end_secs = []
all_probs = []

t0 = time.time()
N_FILES = len(sc_files)

with torch.no_grad():
    for fi, fpath in enumerate(sc_files):
        stem = Path(fpath).stem
        try:
            chunks = file_to_chunks(fpath)
        except Exception as e:
            print(f"WARN: failed to read {stem}: {e}")
            chunks = np.zeros((N_WINDOWS, CHUNK_N), dtype=np.float32)

        wav_t = torch.from_numpy(chunks).unsqueeze(1).to(device)
        mel = mel_transform(wav_t)
        for i in range(mel.size(0)):
            mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
        mel = mel.to(memory_format=torch.channels_last)

        with autocast():
            clip_logits, framewise = model(mel, return_framewise=True)
            frame_max = framewise.max(dim=1).values
            p_clip = torch.sigmoid(clip_logits).float().cpu().numpy()
            p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
        probs_file = 0.5 * p_clip + 0.5 * p_fmax

        probs_file = gaussian_filter1d(probs_file, sigma=GAUSS_SIGMA, axis=0,
                                        mode="nearest").astype(np.float32)

        all_probs.append(probs_file)
        for wi in range(N_WINDOWS):
            all_filenames.append(stem)
            all_start_secs.append(wi * TRAIN_DURATION)
            all_end_secs.append((wi + 1) * TRAIN_DURATION)

        if (fi + 1) % 200 == 0 or fi == N_FILES - 1 or fi == 0:
            elapsed = time.time() - t0
            rate = (fi + 1) / max(elapsed, 1e-6)
            eta = (N_FILES - fi - 1) / max(rate, 1e-6)
            print(f"  [{fi+1:5d}/{N_FILES}] {elapsed:6.1f}s  {rate:5.2f} files/s  ETA {eta/60:5.1f} min")

prob_mat = np.concatenate(all_probs, axis=0).astype(np.float32)
filenames_arr = np.array(all_filenames)
start_secs_arr = np.array(all_start_secs, dtype=np.float32)
end_secs_arr   = np.array(all_end_secs,   dtype=np.float32)
print(f"\nInference: {prob_mat.shape}, mean={prob_mat.mean():.4f}, max={prob_mat.max():.4f} "
      f"in {time.time()-t0:.0f}s")


train_soundscapes: 10658 files
  [    1/10658]    8.0s   0.12 files/s  ETA 1429.5 min
  [  200/10658]   14.8s  13.54 files/s  ETA  12.9 min
  [  400/10658]   21.6s  18.55 files/s  ETA   9.2 min
  [  600/10658]   28.4s  21.16 files/s  ETA   7.9 min
  [  800/10658]   35.2s  22.74 files/s  ETA   7.2 min
  [ 1000/10658]   42.0s  23.83 files/s  ETA   6.8 min
  [ 1200/10658]   48.9s  24.53 files/s  ETA   6.4 min
  [ 1400/10658]   55.8s  25.10 files/s  ETA   6.1 min
  [ 1600/10658]   62.5s  25.62 files/s  ETA   5.9 min
  [ 1800/10658]   69.2s  26.01 files/s  ETA   5.7 min
  [ 2000/10658]   76.2s  26.26 files/s  ETA   5.5 min
  [ 2200/10658]   83.0s  26.50 files/s  ETA   5.3 min
  [ 2400/10658]   89.9s  26.70 files/s  ETA   5.2 min
  [ 2600/10658]   96.7s  26.88 files/s  ETA   5.0 min
  [ 2800/10658]  103.5s  27.04 files/s  ETA   4.8 min
  [ 3000/10658]  110.4s  27.17 files/s  ETA   4.7 min
  [ 3200/10658]  117.2s  27.29 files/s  ETA   4.6 min
  [ 3400/10658]  124.0s  27.42 files/s  ETA   4.4 

In [17]:
# ============================================================
# Cell 16: Post-processing — Power Transform γ=1.2 + Save pseudo CSV
# ============================================================
print(f"Pre-PT: mean={prob_mat.mean():.6f}, max={prob_mat.max():.4f}, "
      f"99%ile={np.percentile(prob_mat, 99):.4f}, "
      f"95%ile={np.percentile(prob_mat, 95):.4f}, "
      f"50%ile={np.percentile(prob_mat, 50):.4f}")

prob_mat = np.power(prob_mat, POWER_GAMMA).astype(np.float32)

print(f"\nPost-PT (γ={POWER_GAMMA}): mean={prob_mat.mean():.6f}, max={prob_mat.max():.4f}, "
      f"99%ile={np.percentile(prob_mat, 99):.4f}, "
      f"95%ile={np.percentile(prob_mat, 95):.4f}, "
      f"50%ile={np.percentile(prob_mat, 50):.4f}")

row_max = prob_mat.max(axis=1)
print(f"\nrow_max stats (diagnostic, NO filter applied):")
print(f"  50%ile: {np.percentile(row_max, 50):.4f}")
print(f"  99%ile: {np.percentile(row_max, 99):.4f}")
print(f"  >=0.05 rows: {(row_max >= 0.05).sum()}/{len(row_max)} "
      f"({(row_max >= 0.05).mean()*100:.1f}%)")

df_p = pd.DataFrame(prob_mat, columns=PRIMARY_LABELS)
df_p.insert(0, "filename",  filenames_arr)
df_p.insert(1, "start_sec", start_secs_arr)
df_p.insert(2, "end_sec",   end_secs_arr)

local_csv = LOCAL_OUT / "pseudo_labels.csv"
df_p.to_csv(local_csv, index=False)
print(f"\nLocal saved: {local_csv}")
print(f"  shape: {df_p.shape}")
print(f"  size:  {local_csv.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  files: {df_p['filename'].nunique()}")

drive_csv = DRIVE_PSEUDO_DIR / "pseudo_labels.csv"
print(f"\nMirroring to Drive: {drive_csv}")
t0 = time.time()
shutil.copy2(str(local_csv), str(drive_csv))
print(f"  done in {time.time()-t0:.1f}s")

pseudo_meta = {
    "backbone": BACKBONE,
    "ckpt": str(PSEUDO_CKPT.name),
    "epoch": state_p.get("epoch"),
    "best_ns22": state_p.get("best_ns22"),
    "best_macro": state_p.get("best_macro"),
    "n_files": int(df_p["filename"].nunique()),
    "n_rows": int(len(df_p)),
    "gauss_sigma": GAUSS_SIGMA,
    "power_gamma": POWER_GAMMA,
}
(DRIVE_PSEUDO_DIR / "pseudo_meta.json").write_text(json.dumps(pseudo_meta, indent=2, default=str))
print(f"  meta written: {DRIVE_PSEUDO_DIR}/pseudo_meta.json")

print(f"\nPseudo phase total time: {(time.time()-PSEUDO_START)/60:.1f} min")


Pre-PT: mean=0.007344, max=0.9997, 99%ile=0.1644, 95%ile=0.0081, 50%ile=0.0004

Post-PT (γ=1.2): mean=0.005683, max=0.9996, 99%ile=0.1146, 95%ile=0.0031, 50%ile=0.0001

row_max stats (diagnostic, NO filter applied):
  50%ile: 0.4674
  99%ile: 0.9950
  >=0.05 rows: 114113/127896 (89.2%)

Local saved: /content/output/pseudo_labels.csv
  shape: (127896, 237)
  size:  391.3 MB
  files: 10658

Mirroring to Drive: /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo/pseudo_labels.csv
  done in 4.5s
  meta written: /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo/pseudo_meta.json

Pseudo phase total time: 6.6 min


---
# Phase 3: Kaggle Dataset upload

R1 weights を `maekeso/birdclef2026-exp017-weights` Dataset として upload。
exp017 R2 (next session) や infer NB から参照可能になる。

[[feedback_kaggle_dataset_403_not_found]]: dataset_view で事前存在チェックして 403/404 罠を回避


In [18]:
# ============================================================
# Cell 17: Upload R1 weights to Kaggle Dataset
# ============================================================
import tempfile

USER  = "maekeso"
SLUG  = "birdclef2026-exp017-weights"
TITLE = "birdclef2026 exp017 weights"

TARGETS = ["ckpt_best_ns22.pth", "ckpt_best_macro.pth", "ckpt_latest.pth", "history.json"]

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    for fn in TARGETS:
        src = DRIVE_OUTPUT_DIR / fn
        if not src.exists():
            print(f"  skip (missing): {fn}")
            continue
        shutil.copy2(str(src), str(td / fn))
        print(f"  staged: {fn}  ({src.stat().st_size/1e6:.1f}MB)")

    meta = {
        "title": TITLE,
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    (td / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    version_notes = f"R1 v2 (LR=3e-4, ep=25) best_ns22={best_ns22:.4f}"
    uploaded = False

    try:
        api.dataset_view(f"{USER}/{SLUG}")
        dataset_exists = True
        print(f"\nDataset {USER}/{SLUG} exists → create new VERSION")
    except Exception:
        dataset_exists = False
        print(f"\nDataset {USER}/{SLUG} not found → CREATE new dataset")

    if dataset_exists:
        try:
            api.dataset_create_version(folder=str(td),
                                        version_notes=version_notes,
                                        dir_mode="zip", quiet=False)
            print(f"\nOK Uploaded (new version) — {version_notes}")
            uploaded = True
        except Exception as e:
            print(f"  dataset_create_version error: {str(e)[:400]}")
    else:
        try:
            api.dataset_create_new(folder=str(td), public=False, dir_mode="zip", quiet=False)
            print(f"\nOK Created (first time)")
            uploaded = True
        except Exception as e:
            print(f"  dataset_create_new error: {str(e)[:400]}")

    if uploaded:
        print(f"\nDataset URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
    else:
        print("\n[WARN] Upload failed — check logs above; you can retry this cell alone.")


  staged: ckpt_best_ns22.pth  (321.8MB)
  staged: ckpt_best_macro.pth  (321.8MB)
  staged: ckpt_latest.pth  (321.8MB)
  staged: history.json  (0.0MB)

Dataset maekeso/birdclef2026-exp017-weights not found → CREATE new dataset

OK Created (first time)

Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp017-weights


In [19]:
# ============================================================
# Cell 18: Final session summary (train + pseudo + upload)
# ============================================================
total_time = time.time() - TRAIN_START
print(f"\n{'='*60}")
print(f"exp017 R1 full session summary")
print(f"{'='*60}")
print(f"  Backbone:             {BACKBONE}")
print(f"  Epochs:               {trained_up_to}/{N_TOTAL_EPOCHS}")
print(f"  Best ns22 AUC:        {best_ns22:.4f}")
print(f"  Best macro AUC:       {best_macro:.4f}")
print(f"  Pseudo CSV:           {DRIVE_PSEUDO_DIR}/pseudo_labels.csv")
print(f"  Kaggle Dataset:       https://www.kaggle.com/datasets/maekeso/birdclef2026-exp017-weights")
print(f"  Total session time:   {total_time/60:.1f} min  ({total_time/3600:.2f}h)")
print(f"\n  Drive output:  {DRIVE_OUTPUT_DIR}")
print(f"  Drive pseudo:  {DRIVE_PSEUDO_DIR}")



exp017 R1 full session summary
  Backbone:             eca_nfnet_l0
  Epochs:               25/25
  Best ns22 AUC:        0.9166
  Best macro AUC:       0.9166
  Pseudo CSV:           /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo/pseudo_labels.csv
  Kaggle Dataset:       https://www.kaggle.com/datasets/maekeso/birdclef2026-exp017-weights
  Total session time:   50.0 min  (0.83h)

  Drive output:  /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1
  Drive pseudo:  /content/drive/MyDrive/kaggle/birdclef2026/output/exp017/r1-pseudo


In [20]:
# ============================================================
# Cell 19: Terminate Colab runtime — save Pro compute units
# ============================================================
# ここまで完走したら A100 を即座に切る。Drive 出力済なので残作業なし。
# Run All で勝手に呼ばれるが、誤実行防止のため確認 print してから切断。
print("All phases complete. Terminating Colab runtime in 5s to free compute units...")
import time as _t
_t.sleep(5)
from google.colab import runtime
runtime.unassign()


All phases complete. Terminating Colab runtime in 5s to free compute units...
